#### SetFit Model Distillation

#### Load in Python Libraries

In [12]:
from datasets import load_from_disk
from setfit import sample_dataset
from setfit import SetFitModel, TrainingArguments
from setfit import DistillationTrainer
import os 
import mlflow

#### Load in Data

In [2]:
dataset = load_from_disk(r"C:\Users\jvhua\OneDrive\Desktop\ISYE-CSE-MGT-6748-Group-1\data\lightcast_chunks_3500.hf")
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'label'],
        num_rows: 1224
    })
    validation: Dataset({
        features: ['id', 'text', 'label'],
        num_rows: 1225
    })
    test: Dataset({
        features: ['id', 'text', 'label'],
        num_rows: 1050
    })
})

#### Set Experiment

In [3]:
mlflow.set_experiment("adeptID")
mlflow.start_run(run_name = "setfit-model-distillation")
mlflow.transformers.autolog()

#### Set Training Test Split

In [4]:
train_dataset = sample_dataset(dataset["train"], label_column="label", num_samples=1224)
eval_dataset = dataset["test"]

c:\Users\jvhua\miniconda3\envs\langchain\Lib\site-packages\setfit\data.py:154: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.apply(lambda x: x.sample(min(num_samples, len(x)), random_state=seed))


#### Choose Teacher and Student Model

In [5]:
teacher_model = SetFitModel.from_pretrained(r"C:\Users\jvhua\OneDrive\Desktop\ISYE-CSE-MGT-6748-Group-1\sefit_model_v6")
student_model = SetFitModel.from_pretrained("sentence-transformers/paraphrase-MiniLM-L3-v2")

c:\Users\jvhua\miniconda3\envs\langchain\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.


#### Set Unlabeled Data for Student to Train on 

In [6]:
unlabeled_train_dataset = dataset["validation"]
unlabeled_train_dataset = unlabeled_train_dataset.remove_columns("label")

#### Training Arguments

In [7]:
distillation_args = TrainingArguments(
    batch_size=8,
    num_epochs=10,
    num_iterations=25,
    max_length = 512,
)

distillation_trainer = DistillationTrainer(
    teacher_model=teacher_model,
    student_model=student_model,
    args=distillation_args,
    train_dataset=unlabeled_train_dataset,
    eval_dataset=eval_dataset,
)

#### Train Model

In [8]:
distillation_trainer.train()

***** Running training *****
  Num unique pairs = 30625
  Batch size = 8
  Num epochs = 10
  Total optimization steps = 38290


  0%|          | 0/38290 [00:00<?, ?it/s]

  0%|          | 0/38290 [00:00<?, ?it/s]

{'embedding_loss': 0.9132, 'learning_rate': 5.223295899712719e-09, 'epoch': 0.0}
{'embedding_loss': 0.7122, 'learning_rate': 2.61164794985636e-07, 'epoch': 0.01}
{'embedding_loss': 0.6659, 'learning_rate': 5.22329589971272e-07, 'epoch': 0.03}
{'embedding_loss': 0.6262, 'learning_rate': 7.834943849569079e-07, 'epoch': 0.04}
{'embedding_loss': 0.4926, 'learning_rate': 1.044659179942544e-06, 'epoch': 0.05}
{'embedding_loss': 0.4081, 'learning_rate': 1.3058239749281798e-06, 'epoch': 0.07}
{'embedding_loss': 0.3778, 'learning_rate': 1.5669887699138158e-06, 'epoch': 0.08}
{'embedding_loss': 0.2208, 'learning_rate': 1.8281535648994516e-06, 'epoch': 0.09}
{'embedding_loss': 0.2381, 'learning_rate': 2.089318359885088e-06, 'epoch': 0.1}
{'embedding_loss': 0.0601, 'learning_rate': 2.3504831548707235e-06, 'epoch': 0.12}
{'embedding_loss': 0.035, 'learning_rate': 2.6116479498563595e-06, 'epoch': 0.13}
{'embedding_loss': 0.0196, 'learning_rate': 2.8728127448419955e-06, 'epoch': 0.14}
{'embedding_los

In [9]:
distillation_metrics = distillation_trainer.evaluate()
print(distillation_metrics)

***** Running evaluation *****


{'accuracy': 0.5552380952380952}


#### Save Distilled Model

In [13]:
model_save_name = "setfit_model_distilled_v3"
folder_path = f"C:\\Users\\jvhua\\OneDrive\\Desktop\\ISYE-CSE-MGT-6748-Group-1\\{model_save_name}"
if not os.path.exists(folder_path):
    os.makedirs(folder_path)
distillation_trainer.model._save_pretrained(f"C:\\Users\\jvhua\\OneDrive\\Desktop\\ISYE-CSE-MGT-6748-Group-1\\{model_save_name}")
mlflow.log_params({'model_distil_name': model_save_name, 'accuracy': distillation_metrics['accuracy']})

In [14]:
mlflow.end_run()